# Workshop 4 - Sistema RAG con NVIDIA NIM + ChromaDB

## Paso 1: Instalar dependencias

In [ ]:
%pip install langchain langchain-nvidia-ai-endpoints langchain-huggingface langchain-community chromadb sentence-transformers python-dotenv -q

## Paso 2: Fase A - Ingesta y creación del Vector Store

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

loader = TextLoader("docs/intro-to-llms-karpathy.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

persist_directory = "db/karpathy_chroma"
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=persist_directory
)
vector_store.persist()
print(f"✅ Vector store creado con {len(docs)} chunks.")

## Paso 3: Fase B - Pipeline RAG

In [ ]:
from dotenv import load_dotenv
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llama_api_key = os.environ.get("LLAMA_API_KEY")

llm = ChatNVIDIA(
    model="meta/llama-4-maverick-17b-128e-instruct",
    api_key=llama_api_key,
    temperature=0.1
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma(
    persist_directory="db/karpathy_chroma",
    embedding_function=embeddings
)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

Context:
{context}

Question: {input}
""")

combine_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(vector_store.as_retriever(), combine_chain)

# Prueba rápida
result = qa_chain.invoke({"input": "What is retrieval augmented generation?"})
print(result["answer"])

## Paso 4: Generar respuestas a las 50 preguntas

In [ ]:
import json

with open("docs/questions.json", "r", encoding="utf-8") as f:
    questions_data = json.load(f)

questions = [item["question"] for item in questions_data]

json_results = []
for i, question in enumerate(questions):
    print(f"[{i+1}/{len(questions)}] {question[:60]}...")
    response = qa_chain.invoke({"input": question})
    json_results.append({
        "question": question,
        "answer": response["answer"],
        "contexts": [doc.page_content for doc in response["context"]]
    })

with open("my_llama_rag_output.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(json_results, indent=4, ensure_ascii=False))

print(f"\n✅ Guardado my_llama_rag_output.json con {len(json_results)} respuestas.")

## Paso 5: Verificar resultados

In [ ]:
with open("my_llama_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[0], indent=1, ensure_ascii=False))

## Deepsek

In [ ]:
from dotenv import load_dotenv
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
deepsek_api_key = os.environ.get("DEEPSEK_API_KEY")

llm = ChatNVIDIA(
    model="deepseek-ai/deepseek-v4-pro",
    api_key=deepsek_api_key,
    temperature=1,
    top_p=0.95,
    max_tokens=16384,
    extra_body={"chat_template_kwargs":{"thinking":False}},
    stream=True
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma(
    persist_directory="db/karpathy_chroma",
    embedding_function=embeddings
)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

Context:
{context}

Question: {input}
""")

combine_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(vector_store.as_retriever(), combine_chain)

# Prueba rápida
result = qa_chain.invoke({"input": "What is retrieval augmented generation?"})
print(result["answer"])

In [ ]:
import json

with open("docs/questions.json", "r", encoding="utf-8") as f:
    questions_data = json.load(f)

questions = [item["question"] for item in questions_data]

json_results = []
for i, question in enumerate(questions):
    print(f"[{i+1}/{len(questions)}] {question[:60]}...")
    response = qa_chain.invoke({"input": question})
    json_results.append({
        "question": question,
        "answer": response["answer"],
        "contexts": [doc.page_content for doc in response["context"]]
    })

with open("my_deepsek_rag_output.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(json_results, indent=4, ensure_ascii=False))

print(f"\n✅ Guardado my_deepsek_rag_output.json con {len(json_results)} respuestas.")

In [ ]:
with open("my_deepsek_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[0], indent=1, ensure_ascii=False))

## Z.ai

In [ ]:
from dotenv import load_dotenv
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
zai_api_key = os.environ.get("ZAI_API_KEY")

llm = ChatNVIDIA(
model="z-ai/glm4.7",
  messages=[{"role":"user","content":""}],
  temperature=1,
  top_p=1,
  max_tokens=16384,
  extra_body={"chat_template_kwargs":{"enable_thinking":True,"clear_thinking":False}},
  stream=True,
  api_key=zai_api_key
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma(
    persist_directory="db/karpathy_chroma",
    embedding_function=embeddings
)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

Context:
{context}

Question: {input}
""")

combine_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(vector_store.as_retriever(), combine_chain)

# Prueba rápida
result = qa_chain.invoke({"input": "What is retrieval augmented generation?"})
print(result["answer"])

In [ ]:
import json

with open("docs/questions.json", "r", encoding="utf-8") as f:
    questions_data = json.load(f)

questions = [item["question"] for item in questions_data]

json_results = []
for i, question in enumerate(questions):
    print(f"[{i+1}/{len(questions)}] {question[:60]}...")
    response = qa_chain.invoke({"input": question})
    json_results.append({
        "question": question,
        "answer": response["answer"],
        "contexts": [doc.page_content for doc in response["context"]]
    })

with open("my_zai_rag_output.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(json_results, indent=4, ensure_ascii=False))

print(f"\n✅ Guardado my_zai_rag_output.json con {len(json_results)} respuestas.")

In [ ]:
with open("my_zai_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[0], indent=1, ensure_ascii=False))